# Stage 11 V2a — Frozen candidate packaging

Bu notebook **eğitim yapmaz**. Drive'daki dondurulmuş V2a `best.pt` dosyasının SHA-256 kimliğini doğrular, model ağırlıklarını değiştirmeden CPU üzerinde deterministik smoke test yapar ve özel Drive alanına TorchScript aday paketi + kanıt JSON'u yazar.

Held-out veri okunmaz. Production inference ve Stage 12 açılmaz.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys, json
REPO = Path('/content/st-score-restore-engine')
REF = os.environ.get('ST_SCORE_RESTORE_REF', 'main')
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','--branch',REF,'https://github.com/khfy7wpr5p-maker/st-score-restore-engine.git',str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin',REF,'--depth','1'], check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'], check=True)
import torch
CHECKPOINT = Path('/content/drive/MyDrive/ST_SCORE_RESTORE_STAGE11_TRAINING_OUTPUT/deepscoresv2_dense_residual_unet_v2a_symbol_preservation/best.pt')
print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__)
print('Repo commit:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip())
print('Checkpoint exists:', CHECKPOINT.exists())
if not CHECKPOINT.exists(): raise FileNotFoundError(CHECKPOINT)
print('Preflight: OK — packaging CPU üzerinde çalışacak.')


## 1. Checkpoint doğrulama + paketleme + smoke test
Bu işlem 512×512 tek bir deterministik probe ile CPU üzerinde yapılır. Ağırlıklar değişirse, SHA uyuşmazsa veya TorchScript sonucu eager modelle eşleşmezse fail-closed durur.


In [ ]:
cmd = [sys.executable, '-u', str(REPO/'tools'/'stage11_v2a_candidate_packaging.py')]
print('Çalıştırılıyor:', ' '.join(cmd))
subprocess.run(cmd, check=True)


## 2. Sonuç kanıtı


In [ ]:
OUT = Path('/content/drive/MyDrive/ST_SCORE_RESTORE_STAGE11_PACKAGING/deepscoresv2_dense_v2a_candidate')
EVIDENCE = OUT/'candidate_package_evidence.v1.json'
PACKAGE = OUT/'v2a_candidate_512.torchscript.pt'
print('Package:', PACKAGE, 'exists=', PACKAGE.exists(), 'bytes=', PACKAGE.stat().st_size if PACKAGE.exists() else None)
print('Evidence:', EVIDENCE)
if not EVIDENCE.exists(): raise FileNotFoundError(EVIDENCE)
payload = json.loads(EVIDENCE.read_text())
print(json.dumps(payload, indent=2, ensure_ascii=False))
assert payload['status'] == 'completed'
assert payload['weightsMutated'] is False
assert payload['heldOutAccessed'] is False
assert payload['authorization']['productionInferenceAuthorized'] is False
print('\nPACKAGING SMOKE TEST: PASS')
